<a href="https://colab.research.google.com/github/RachmadAprisandhy/PembelajaranMesin/blob/main/JS09/JS09_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

instalasi yang dibutuhkan

In [1]:
!pip install -q annoy faiss-cpu hnswlib pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


import library

In [2]:
import numpy as np
import pandas as pd
import time

from annoy import AnnoyIndex
import faiss
import hnswlib

kode percobaan

In [3]:

import numpy as np
import pandas as pd
import time

from annoy import AnnoyIndex
import faiss
import hnswlib

# ==========================================
# KONFIGURASI EKSPERIMEN
# ==========================================

np.random.seed(42)

DATA_CONFIGS = [
    (1000, 2),
    (1000000, 5)
]

METRICS = ["euclidean", "cosine"]

K = 10
N_QUERIES = 100

results = []

# ==========================================
# FUNGSI NORMALISASI COSINE
# ==========================================

def normalize_vectors(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, 1e-10)


# ==========================================
# EKSPERIMEN
# ==========================================

for n_data, dim in DATA_CONFIGS:

    print("\n" + "=" * 60)
    print(f"DATASET: {n_data:,} DATA | {dim} DIMENSI")
    print("=" * 60)

    # Generate dataset
    rng = np.random.default_rng(42)

    X = rng.standard_normal(
        (n_data, dim)
    ).astype(np.float32)

    query = rng.standard_normal(
        (N_QUERIES, dim)
    ).astype(np.float32)

    for metric in METRICS:

        print(f"\n--- METRIC: {metric.upper()} ---")

        # Siapkan data berdasarkan metric
        if metric == "cosine":
            X_test = normalize_vectors(X)
            Q_test = normalize_vectors(query)
        else:
            X_test = X
            Q_test = query

        # ==================================
        # 1. FAISS - EXACT BASELINE
        # ==================================

        print("Running FAISS...")

        if metric == "euclidean":
            faiss_index = faiss.IndexFlatL2(dim)
        else:
            faiss_index = faiss.IndexFlatIP(dim)

        start = time.perf_counter()
        faiss_index.add(X_test)
        faiss_build = time.perf_counter() - start

        start = time.perf_counter()
        gt_distances, gt_indices = faiss_index.search(
            Q_test, K
        )
        faiss_query = time.perf_counter() - start

        # Ground truth untuk menghitung Recall@K
        ground_truth = gt_indices

        results.append({
            "Data": n_data,
            "Dimensi": dim,
            "Metric": metric,
            "Algorithm": "FAISS",
            "Build (s)": faiss_build,
            "Query (ms)": faiss_query / N_QUERIES * 1000,
            "Recall@10 (%)": 100.0
        })

        # ==================================
        # 2. ANNOY
        # ==================================

        print("Running Annoy...")

        annoy_metric = (
            "euclidean" if metric == "euclidean"
            else "angular"
        )

        annoy_index = AnnoyIndex(dim, annoy_metric)

        start = time.perf_counter()

        for i in range(n_data):
            annoy_index.add_item(i, X_test[i])

        annoy_index.build(10)

        annoy_build = time.perf_counter() - start

        start = time.perf_counter()

        annoy_results = []

        for q in Q_test:
            ids = annoy_index.get_nns_by_vector(
                q.tolist(), K
            )
            annoy_results.append(ids)

        annoy_query = time.perf_counter() - start

        # Recall@K
        recalls = []

        for i in range(N_QUERIES):
            recall = len(
                set(annoy_results[i]) &
                set(ground_truth[i])
            ) / K

            recalls.append(recall)

        results.append({
            "Data": n_data,
            "Dimensi": dim,
            "Metric": metric,
            "Algorithm": "ANNOY",
            "Build (s)": annoy_build,
            "Query (ms)": annoy_query / N_QUERIES * 1000,
            "Recall@10 (%)": np.mean(recalls) * 100
        })

        # ==================================
        # 3. HNSWlib
        # ==================================

        print("Running HNSWlib...")

        hnsw_metric = (
            "l2" if metric == "euclidean"
            else "cosine"
        )

        hnsw_index = hnswlib.Index(
            space=hnsw_metric,
            dim=dim
        )

        start = time.perf_counter()

        hnsw_index.init_index(
            max_elements=n_data,
            ef_construction=200,
            M=16
        )

        hnsw_index.add_items(X_test)

        hnsw_build = time.perf_counter() - start

        hnsw_index.set_ef(50)

        start = time.perf_counter()

        hnsw_labels, hnsw_distances = hnsw_index.knn_query(
            Q_test, k=K
        )

        hnsw_query = time.perf_counter() - start

        # Recall@K
        recalls = []

        for i in range(N_QUERIES):
            recall = len(
                set(hnsw_labels[i]) &
                set(ground_truth[i])
            ) / K

            recalls.append(recall)

        results.append({
            "Data": n_data,
            "Dimensi": dim,
            "Metric": metric,
            "Algorithm": "HNSW",
            "Build (s)": hnsw_build,
            "Query (ms)": hnsw_query / N_QUERIES * 1000,
            "Recall@10 (%)": np.mean(recalls) * 100
        })

        print("Experiment completed!")

        # Bersihkan index sebelum konfigurasi selanjutnya
        del annoy_index
        del hnsw_index
        del faiss_index


# ==========================================
# HASIL EKSPERIMEN
# ==========================================

df_results = pd.DataFrame(results)

print("\n========== HASIL KESELURUHAN ==========")

display(
    df_results.round(4)
)

# Simpan hasil ke CSV
df_results.to_csv(
    "hasil_eksperimen_ann.csv",
    index=False
)

print("\nHasil berhasil disimpan: hasil_eksperimen_ann.csv")



DATASET: 1,000 DATA | 2 DIMENSI

--- METRIC: EUCLIDEAN ---
Running FAISS...
Running Annoy...
Running HNSWlib...
Experiment completed!

--- METRIC: COSINE ---
Running FAISS...
Running Annoy...
Running HNSWlib...
Experiment completed!

DATASET: 1,000,000 DATA | 5 DIMENSI

--- METRIC: EUCLIDEAN ---
Running FAISS...
Running Annoy...
Running HNSWlib...
Experiment completed!

--- METRIC: COSINE ---
Running FAISS...
Running Annoy...
Running HNSWlib...
Experiment completed!

========== HASIL KESELURUHAN ==========


,Data,Dimensi,Metric,Algorithm,Build (s),Query (ms),Recall@10 (%)
0,1000,2,euclidean,FAISS,0.0001,0.1506,100.0
1,1000,2,euclidean,ANNOY,0.0574,0.0360,99.3
2,1000,2,euclidean,HNSW,0.1527,0.0404,100.0
3,1000,2,cosine,FAISS,0.0000,0.0673,100.0
4,1000,2,cosine,ANNOY,0.0609,0.0842,99.9
5,1000,2,cosine,HNSW,0.1145,0.0264,100.0
6,1000000,5,euclidean,FAISS,0.0242,11.5910,100.0
7,1000000,5,euclidean,ANNOY,22.6383,0.0614,94.3
8,1000000,5,euclidean,HNSW,188.0168,0.0488,100.0
9,1000000,5,cosine,FAISS,0.0178,10.4821,100.0



Hasil berhasil disimpan: hasil_eksperimen_ann.csv


tabel hasil

In [4]:
# Tabel waktu pembangunan index
build_table = df_results.pivot_table(
    index=["Data", "Dimensi", "Metric"],
    columns="Algorithm",
    values="Build (s)"
)

print("WAKTU BUILD INDEX (DETIK)")
display(build_table.round(4))

# Tabel waktu pencarian
query_table = df_results.pivot_table(
    index=["Data", "Dimensi", "Metric"],
    columns="Algorithm",
    values="Query (ms)"
)

print("WAKTU QUERY (MILIDETIK)")
display(query_table.round(4))

# Tabel akurasi
recall_table = df_results.pivot_table(
    index=["Data", "Dimensi", "Metric"],
    columns="Algorithm",
    values="Recall@10 (%)"
)

print("RECALL@10 (PERSEN)")
display(recall_table.round(2))

WAKTU BUILD INDEX (DETIK)


Algorithm                    ANNOY   FAISS      HNSW
Data    Dimensi Metric                              
1000    2       cosine      0.0609  0.0000    0.1145
                euclidean   0.0574  0.0001    0.1527
1000000 5       cosine     29.2136  0.0178  169.4126
                euclidean  22.6383  0.0242  188.0168

WAKTU QUERY (MILIDETIK)


Algorithm                   ANNOY    FAISS    HNSW
Data    Dimensi Metric                            
1000    2       cosine     0.0842   0.0673  0.0264
                euclidean  0.0360   0.1506  0.0404
1000000 5       cosine     0.0627  10.4821  0.0366
                euclidean  0.0614  11.5910  0.0488

RECALL@10 (PERSEN)


Algorithm                  ANNOY  FAISS   HNSW
Data    Dimensi Metric                        
1000    2       cosine      99.9  100.0  100.0
                euclidean   99.3  100.0  100.0
1000000 5       cosine      97.2  100.0  100.0
                euclidean   94.3  100.0  100.0